# Project Touchdown, stage 2: change the points

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArseniyCherednichenko/project-touchdown/blob/main/notebooks/stage2_change_the_points.ipynb)

In [stage 1](https://colab.research.google.com/github/ArseniyCherednichenko/project-touchdown/blob/main/notebooks/stage1_land_a_rocket.ipynb)
an AI learned to land a rocket only from points. Here you find out how much
those points matter: same AI, same training, five different sets of points.
Then you design your own.

Run the cells top to bottom with Shift + Enter.

Part of [Project Touchdown](https://github.com/ArseniyCherednichenko/project-touchdown)
by Arsenii Cherednichenko.

## 1. Install the tools

In [ ]:
!pip install -q "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg

## 2. How the points really work

This is Lunar Lander's scoring, taken from its source code. Every frame it
computes one number, the **shaping**, that says how good the current
situation is:

```
shaping = -100 * distance to the pad
          -100 * speed
          -100 * tilt
          + 10 per leg on the ground
```

The trick: the AI does **not** get the shaping itself. It gets the **change**
since the last frame. Moving closer to the pad, slowing down or leveling out
earns points. Drifting away loses them. Just sitting somewhere close earns
nothing, because nothing changed.

On top of that:
- main engine: **-0.3** per frame, side engines: **-0.03** per frame
- at the end: **-100** for a crash, **+100** for coming to rest

Below is the same scoring written out, but with every number as a setting
you can change.

In [ ]:
import gymnasium as gym
import numpy as np

DEFAULT = dict(distance=100, speed=100, tilt=100, legs=10,
               main_engine=0.30, side_engine=0.03, crash=-100, rest=100)

class Points(gym.Wrapper):
    """Lunar Lander with your own points. Unchanged settings = original game."""
    def __init__(self, env, **changes):
        super().__init__(env)
        self.w = {**DEFAULT, **changes}

    def shaping(self, s):
        w = self.w
        return (-w["distance"] * np.hypot(s[0], s[1])   # distance to pad
                - w["speed"] * np.hypot(s[2], s[3])     # speed
                - w["tilt"] * abs(s[4])                 # tilt
                + w["legs"] * (s[6] + s[7]))            # legs on the ground

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev = self.shaping(obs)
        return obs, info

    def step(self, action):
        obs, original, terminated, truncated, info = self.env.step(action)
        info["original_points"] = original   # the real game's points, for fair tests
        now = self.shaping(obs)
        points = now - self.prev              # reward for IMPROVING
        self.prev = now
        if action == 2:
            points -= self.w["main_engine"]
        elif action in (1, 3):
            points -= self.w["side_engine"]
        if terminated:
            crashed = self.env.unwrapped.game_over or abs(obs[0]) >= 1.0
            points = self.w["crash"] if crashed else self.w["rest"]
            info["outcome"] = "crashed" if crashed else "landed"
        elif truncated:
            info["outcome"] = "hovered"
        return obs, points, terminated, truncated, info

def make(render=False, **changes):
    env = gym.make("LunarLander-v3", render_mode="rgb_array" if render else None)
    return Points(env, **changes)

Quick check that the copy is exact: with no changes, our points should match
the game's own points on every frame.

In [ ]:
env = make()
biggest_difference = 0
for seed in range(5):
    obs, _ = env.reset(seed=seed)
    done = False
    while not done:
        obs, points, term, trunc, info = env.step(env.action_space.sample())
        biggest_difference = max(biggest_difference, abs(points - info["original_points"]))
        done = term or trunc
print(f"Biggest difference: {biggest_difference:.6f}  (0 means identical)")

## 3. Five sets of points

I trained one AI on each of these, 1,000,000 steps each, with the exact same
settings as stage 1. Only the points differ.

In [ ]:
VARIANTS = {
    "default":           dict(),                                    # the original game
    "free_fuel":         dict(main_engine=0, side_engine=0),        # engines cost nothing
    "expensive_fuel":    dict(main_engine=3.0, side_engine=0.3),    # 10x fuel cost
    "no_crash_penalty":  dict(crash=0),                             # crashing costs nothing
    "only_final_points": dict(distance=0, speed=0, tilt=0, legs=0,  # no hints at all,
                              main_engine=0, side_engine=0),        # only +100 / -100
}

**Before you scroll on, guess.** For each one: does it still land, crash, or
hover forever? Write your guesses down somewhere.

## 4. Load my five trained AIs and watch them

In [ ]:
import urllib.request, base64, tempfile, imageio
from IPython.display import HTML, display
from stable_baselines3 import PPO

models = {}
for name in VARIANTS:
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/ArseniyCherednichenko/project-touchdown/main/models/variants/{name}.zip", f"{name}.zip")
    models[name] = PPO.load(f"{name}.zip")
print("Loaded:", ", ".join(models))

def show(frames, fps=50):
    path = tempfile.mktemp(suffix=".mp4")
    imageio.mimsave(path, frames, fps=fps, macro_block_size=8)
    data = base64.b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width="420" controls autoplay loop '
                f'src="data:video/mp4;base64,{data}"></video>')

def fly(model, seed, render=False):
    """One flight, scored with the ORIGINAL points so every AI is judged the same."""
    env = make(render=render)
    obs, _ = env.reset(seed=seed)
    frames = [env.render()] if render else []
    score, engine_frames, done = 0.0, 0, False
    while not done:
        action = model.predict(obs, deterministic=True)[0]
        obs, _, term, trunc, info = env.step(action)
        score += info["original_points"]
        engine_frames += action == 2
        if render:
            frames.append(env.render())
        done = term or trunc
    return dict(frames=frames, score=score, outcome=info["outcome"],
                engine_seconds=engine_frames / 50)

In [ ]:
for name, model in models.items():
    result = fly(model, seed=1004, render=True)
    print(f"{name}: {result['outcome']}, score {result['score']:.0f}")
    display(show(result["frames"][::2]))   # 2x speed

## 5. A fair test

One flight can be luck. Here every AI flies the same 20 starts, scored with
the original points.

In [ ]:
TEST_FLIGHTS = 20

print(f"{'points':<19}{'score':>6}{'landed':>8}{'crashed':>9}{'hovered':>9}{'engine s':>10}")
for name, model in models.items():
    runs = [fly(model, seed=1000 + i) for i in range(TEST_FLIGHTS)]
    share = lambda o: sum(r["outcome"] == o for r in runs) / TEST_FLIGHTS
    print(f"{name:<19}{np.mean([r['score'] for r in runs]):>6.0f}"
          f"{share('landed'):>8.0%}{share('crashed'):>9.0%}{share('hovered'):>9.0%}"
          f"{np.mean([r['engine_seconds'] for r in runs]):>10.1f}")

## 6. Why each one did that

**Free fuel.** Still lands, but burns almost twice as long. Nothing told it
to save fuel anymore, so it doesn't.

**Fuel 10x expensive.** Every frame of braking now costs 3 points. Landing
safely is worth +100, but braking for 3 seconds costs about 450. So from the
AI's point of view, braking is a bad deal, and it barely does it. It falls
and crashes. It did exactly what the points said.

**Crashing is free.** Surprise: it still lands. A crash ends the flight, and
then the +100 for landing can never come. Plus all the small "getting closer,
slowing down" points still reward a careful approach. The -100 turned out to
barely matter.

**No hints.** It never learns. A random pilot almost never lands by luck, so
it almost never sees a +100, so there is nothing to learn from. The small
points along the way are what make learning possible at all. This is called
a **sparse reward** problem, and it is one of the big problems in AI.

**The lesson:** the AI does not do what you want. It does what you reward.
Designing the points is most of the work.

## 7. Design your own points

Change the numbers below, train, and see what your AI does. Ideas:
- `tilt=1000`: does it become obsessed with staying level?
- `legs=0`: does it still bother to touch down gently?
- `rest=1000`: does a huge landing bonus fix the expensive fuel problem?
  (combine with `main_engine=3.0`)

Training 1,000,000 steps took 23 minutes on free Colab in stage 1 (measured).
This one should be similar.

In [ ]:
MY_POINTS = dict(tilt=1000)
TRAIN_STEPS = 1_000_000

from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make(**MY_POINTS), n_envs=16)
mine = PPO("MlpPolicy", train_env, n_steps=1024, batch_size=64, gae_lambda=0.98,
           gamma=0.999, n_epochs=4, ent_coef=0.01, seed=0)
mine.learn(total_timesteps=TRAIN_STEPS)

runs = [fly(mine, seed=1000 + i) for i in range(TEST_FLIGHTS)]
print(f"Your points {MY_POINTS}: average {np.mean([r['score'] for r in runs]):.0f}, "
      f"landed {sum(r['outcome'] == 'landed' for r in runs)} of {TEST_FLIGHTS}")
result = fly(mine, seed=1004, render=True)
show(result["frames"][::2])

## What I learned

- An AI optimizes the points, not your intention. Make braking expensive and
  you get a rocket that is afraid to brake.
- The hints along the way matter more than the big reward at the end. Without
  them, it never finds the reward in the first place.
- Some rules you think are essential, like punishing crashes, barely matter.
  You only find out by testing.

Found a set of points that beats the original? [Tell me](https://github.com/ArseniyCherednichenko/project-touchdown/issues).

Next, in stage 3: my own rocket, one that can't hover.